In [ ]:
%load_ext cudf.pandas

In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:

from pathlib import Path
import numpy as np
import os
from utils.benchmarks import BENCHMARKS_TO_PATHS

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    import pandas as pd
import time

In [ ]:
%%time
### cell 0 ###

benchmark_name = "feedback3-eda-hf-custom-trainer-sift"
train_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "feedback-prize-english-language-learning" / "train.csv"
)
test_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "feedback-prize-english-language-learning" / "test.csv"
)

factor = 500
train_df = pd.concat([train_df] * factor)
test_df = pd.concat([test_df] * factor)
train_df.info()

In [ ]:
%%time
### cell 1 ###

train_df.head()

In [ ]:
%%time
### cell 2 ###

test_df.head()

In [ ]:
%%time
### cell 3 ###

len(train_df), len(test_df)

In [ ]:
%%time
### cell 4 ###

LABEL_COLUMNS = [
    "cohesion",
    "syntax",
    "vocabulary",
    "phraseology",
    "grammar",
    "conventions",
]

In [ ]:
%%time
### cell 5 ###

texts = train_df.sample(n=4, random_state=420)

In [ ]:
%%time
### cell 6 ###

train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
lowest_df = train_df.sort_values("total_score").head(4)

In [ ]:
%%time
### cell 7 ###

train_df["total_score"] = train_df[LABEL_COLUMNS].sum(axis=1)
highest_df = train_df.sort_values("total_score", ascending=False).head(4)

In [ ]:
%%time
### cell 8 ###
# use a fully‐vectorized GPU string split + list length, then cast to int64 to match the original dtype
train_df['word_count'] = train_df.full_text.str.split().list.len().astype('int64')

In [ ]:
%%time
### cell 9 ###

train_df["word_count"].mean()

In [ ]:
%%time
### cell 10 ###

train_df["word_count"].max()